## Jefferies PnL, reduce markouts

## Data Prep

In [1]:
import numpy as np
import pandas as pd

In [2]:
raw = pd.read_excel('RFQ US Rates - Bonds Only (July - Sept 24) - Blockhouse.xlsx')
raw

,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,SettlementDays,SettlementDate,SettlementStops,RFQ CreateTime,RFQ ClosedTime,AnAutoReplyRFQs,First Response Time (Sec),TradeSizeBucket,Sector,MaturityBucket
0,2024-09-30,66FA7D45432800D20002,Client8,Done,OFFTHERUN,100000000,100000000,100000000,100000000,100000000,...,1,2024-10-01,T+1,2024-09-30 06:28:00,2024-09-30 06:28:00,1,0.03,50+M,5-7 YR,5y-7y
1,2024-09-30,TRSY_20240930_15961,Client9,Done,"OFFTHERUN,DOUBLEOLDS",2100000,2100000,2100000,2100000,2100000,...,1,2024-10-01,T+1,2024-09-30 16:00:00,2024-09-30 16:00:00,1,0.95,1-5M,7-10 YR,7y-10y
2,2024-09-30,TRSY_20240930_314,Client9,Done,ONTHERUN,53000000,53000000,53000000,53000000,53000000,...,1,2024-10-01,T+1,2024-09-30 15:46:00,2024-09-30 15:46:00,1,0.07,50+M,7-10 YR,7y-10y
3,2024-09-30,66FAEA57432800020001,Client41,Done,OFFTHERUN,500000,500000,500000,500000,500000,...,1,2024-10-01,T+1,2024-09-30 14:14:00,2024-09-30 14:14:00,1,0.03,500K-1M,0-1 YR,<18mos
4,2024-09-30,66FAC35D455C00210006,Client139,Done,OFFTHERUN,1058000,1058000,1058000,1058000,1058000,...,1,2024-10-01,T+1,2024-09-30 11:27:00,2024-09-30 11:27:00,1,0.03,1-5M,5-7 YR,3y-5y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40825,2024-07-01,E682EBF3454C00010001,Client736,CustomerTimeOut,OFFTHERUN,0,0,0,0,0,...,1,2024-07-02,T+1,2024-07-01 13:48:00,2024-07-01 13:50:00,1,0.03,10-25M,5-7 YR,3y-5y
40826,2024-07-01,E682D1C745CC000A0001,Client736,CustomerTimeOut,OFFTHERUN,0,0,0,0,0,...,1,2024-07-02,T+1,2024-07-01 11:56:00,2024-07-01 11:58:00,1,0.06,25-50M,5-7 YR,3y-5y
40827,2024-07-01,E682C785454C00010001,Client736,CustomerTimeOut,OFFTHERUN,0,0,0,0,0,...,1,2024-07-02,T+1,2024-07-01 11:13:00,2024-07-01 11:14:00,1,0.03,25-50M,5-7 YR,3y-5y
40828,2024-07-01,E682AFF2454C000C0001,Client736,CustomerTimeOut,"OFFTHERUN,DOUBLEOLDS",0,0,0,0,0,...,1,2024-07-02,T+1,2024-07-01 09:32:00,2024-07-01 09:34:00,1,0.04,10-25M,5-7 YR,3y-5y


In [3]:
raw.columns

Index(['EventDate', 'Id', 'Client', 'RFQStatus', 'InstrumentSubGroup',
       'Mkt Traded Vol', 'Market Traded Vol (USD)', 'Our Traded Vol',
       'Our Traded Vol (USD)', 'TiedWonVol', 'Num of Dealers', 'Trader',
       'Sales', 'Covered Vol', 'ActionStr', 'Tier', 'InstrumentCode',
       'InstrumentDescription', 'AssetClass', 'Market', 'LegNo',
       'Maturity Date', 'RFQNumberOfQuotes', 'Buy/Sell', 'Best Bid Price',
       'Mid Price', 'Best Ask Price', 'Deal Value', 'AwayFromMid',
       'TickIdentifier', 'Deal Spread', 'Cover', 'CoverPnL', 'SettlementDays',
       'SettlementDate', 'SettlementStops', 'RFQ CreateTime', 'RFQ ClosedTime',
       'AnAutoReplyRFQs', 'First Response Time (Sec)', 'TradeSizeBucket',
       'Sector', 'MaturityBucket'],
      dtype='object')

In [4]:
raw['ActionStr'].value_counts()

ActionStr
TradedAway           22617
Covered               6658
CustRejectedQuote     3406
Expired               2400
TiedTradedAway        2161
CustAcceptedQuote     1491
DealerAcceptOrder     1364
CoverTied              659
DealerReject            74
Name: count, dtype: int64

In [5]:
raw.Tier.value_counts()

Tier
TIER2           28354
TIER1            5004
MANUAL           3337
TierINTERNAL     2827
TIER9             793
TierGOLD          302
TIER3             182
UNKNOWN            31
Name: count, dtype: int64

In [108]:
raw.MaturityBucket.value_counts()

MaturityBucket
10-30y    10499
<18mos     8457
7y-10y     8065
3y-5y      6827
5y-7y      2747
18m-2y     2218
2y-3y      2017
Name: count, dtype: int64

In [6]:
valid_tiers = ['TIER1', 'TIER2', 'TIER3', 'TIER9', 'TierGOLD', 'TierINTERNAL', 'MANUAL']

# Apply filters
test = raw[
    (raw['Our Traded Vol'] != 0) & 
    (raw['Tier'].isin(valid_tiers))
].copy()

# view = ['EventDate','Market Traded Vol (USD)','Our Traded Vol (USD)','Tier','Buy/Sell','Best Bid Price','Best Ask Price','Mid Price','Deal Value','Deal Spread','Cover','CoverPnL','Sector']
test

,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,SettlementDays,SettlementDate,SettlementStops,RFQ CreateTime,RFQ ClosedTime,AnAutoReplyRFQs,First Response Time (Sec),TradeSizeBucket,Sector,MaturityBucket
0,2024-09-30,66FA7D45432800D20002,Client8,Done,OFFTHERUN,100000000,100000000,100000000,100000000,100000000,...,1,2024-10-01,T+1,2024-09-30 06:28:00,2024-09-30 06:28:00,1,0.03,50+M,5-7 YR,5y-7y
1,2024-09-30,TRSY_20240930_15961,Client9,Done,"OFFTHERUN,DOUBLEOLDS",2100000,2100000,2100000,2100000,2100000,...,1,2024-10-01,T+1,2024-09-30 16:00:00,2024-09-30 16:00:00,1,0.95,1-5M,7-10 YR,7y-10y
2,2024-09-30,TRSY_20240930_314,Client9,Done,ONTHERUN,53000000,53000000,53000000,53000000,53000000,...,1,2024-10-01,T+1,2024-09-30 15:46:00,2024-09-30 15:46:00,1,0.07,50+M,7-10 YR,7y-10y
3,2024-09-30,66FAEA57432800020001,Client41,Done,OFFTHERUN,500000,500000,500000,500000,500000,...,1,2024-10-01,T+1,2024-09-30 14:14:00,2024-09-30 14:14:00,1,0.03,500K-1M,0-1 YR,<18mos
4,2024-09-30,66FAC35D455C00210006,Client139,Done,OFFTHERUN,1058000,1058000,1058000,1058000,1058000,...,1,2024-10-01,T+1,2024-09-30 11:27:00,2024-09-30 11:27:00,1,0.03,1-5M,5-7 YR,3y-5y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40073,2024-07-01,E682C6A157D0001B0001,Client736,Done,ONTHERUN,1000000,1000000,1000000,1000000,0,...,1,2024-07-02,T+1,2024-07-01 11:09:00,2024-07-01 11:09:00,1,0.04,1-5M,5-7 YR,3y-5y
40074,2024-07-01,TRSY_20240701_9910,Client775,Done,"OFFTHERUN,OLDBONDS",3660000,3660000,3660000,3660000,0,...,1,2024-07-02,T+1,2024-07-01 15:57:00,2024-07-01 15:57:00,0,0.07,1-5M,15-30 YR,10-30y
40075,2024-07-01,TRSY_20240701_5800,Client775,Done,ONTHERUN,9630000,9630000,9630000,9630000,0,...,1,2024-07-02,T+1,2024-07-01 13:15:00,2024-07-01 13:15:00,0,0.06,5-10M,7-10 YR,7y-10y
40076,2024-07-01,TRSY_20240701_9909,Client775,Done,ONTHERUN,2300000,2300000,2300000,2300000,0,...,1,2024-07-02,T+1,2024-07-01 11:58:00,2024-07-01 11:58:00,0,0.06,1-5M,15-30 YR,10-30y


In [7]:
test.Tier.value_counts()

Tier
TIER2           1698
TierINTERNAL     723
TIER1            212
TierGOLD         104
TIER9             46
TIER3             35
MANUAL            33
Name: count, dtype: int64

## PnL for bid ask spread (no inventory control)

Gain of single trade
$$ G_{T} = v_0 \epsilon_{t} \left(m_{t} + \epsilon_{t} \frac{S_{t}}{2} - m_{T} \right)  $$


Expected gain of market-maker between time $0$ and $T$

$$ \mathbb{E}[G_T] = v_0 \left( \sum_{t=0}^{T-1} \mathbb{E} \left[ \theta_t \varepsilon_t \left( m_t + \varepsilon_t \frac{S_t}{2} - m_T \right) + \theta_t \omega \right] \right) $$

where $G_{T}$ is gain at time $T$,

$v_{0}$ is volume,

$S_{t}$ is the spread

$\epsilon$ is order sign,

$m_{t}$ is mid-price at time $t$,

$\theta_t$ is execution indicator,

$\omega$ is additional payment

We assume that there are no additional payments like processing or transaction costs. We also assume that every trade is executed. Therefore, $\theta = 1$ and $\omega = 0$

$$ \mathbb{E}[G_T] = v_0 \left( \sum_{t=0}^{T-1} \mathbb{E} \left[\varepsilon_t \left( m_t + \varepsilon_t \frac{S_{t}}{2} - m_{T} \right) \right] \right) $$

In [8]:
test['Sign'] = test['Buy/Sell'].apply(lambda x: 1 if x == 'Buy' else -1)
test['Sign'].value_counts()

Sign
-1    1557
 1    1294
Name: count, dtype: int64

In [ ]:
def pnl(df):
    df_calc = df.copy() 
    # Calculate spread
    df_calc['spread'] = ((df_calc['Best Ask Price'] - df_calc['Best Bid Price'])/df_calc['Deal Value']) * 10000
    
    # Calculate PnL per trade using the formula
    df_calc['PnL'] = (
        df_calc['Our Traded Vol (USD)'] *  # v_0
        df_calc['Sign'] * (             # ε_t
            df_calc['Mid Price'] +         # m_t
            df_calc['Sign'] * ((df_calc['spread'] / 2) * df_calc['Deal Value'] / 10000) - # ε_t * S_t/2
            df_calc['Mid Price']           # m_T (same as m_t for individual trades)
        )
    )
    
    return df_calc

In [10]:
result1 = pnl(test)
result1

,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,RFQ CreateTime,RFQ ClosedTime,AnAutoReplyRFQs,First Response Time (Sec),TradeSizeBucket,Sector,MaturityBucket,Sign,spread,PnL
0,2024-09-30,66FA7D45432800D20002,Client8,Done,OFFTHERUN,100000000,100000000,100000000,100000000,100000000,...,2024-09-30 06:28:00,2024-09-30 06:28:00,1,0.03,50+M,5-7 YR,5y-7y,-1,2.354141,1.171875e+06
1,2024-09-30,TRSY_20240930_15961,Client9,Done,"OFFTHERUN,DOUBLEOLDS",2100000,2100000,2100000,2100000,2100000,...,2024-09-30 16:00:00,2024-09-30 16:00:00,1,0.95,1-5M,7-10 YR,7y-10y,-1,2.303971,2.460938e+04
2,2024-09-30,TRSY_20240930_314,Client9,Done,ONTHERUN,53000000,53000000,53000000,53000000,53000000,...,2024-09-30 15:46:00,2024-09-30 15:46:00,1,0.07,50+M,7-10 YR,7y-10y,-1,0.783116,2.070312e+05
3,2024-09-30,66FAEA57432800020001,Client41,Done,OFFTHERUN,500000,500000,500000,500000,500000,...,2024-09-30 14:14:00,2024-09-30 14:14:00,1,0.03,500K-1M,0-1 YR,<18mos,-1,0.784745,1.953125e+03
4,2024-09-30,66FAC35D455C00210006,Client139,Done,OFFTHERUN,1058000,1058000,1058000,1058000,1058000,...,2024-09-30 11:27:00,2024-09-30 11:27:00,1,0.03,1-5M,5-7 YR,3y-5y,1,1.537811,8.265625e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40073,2024-07-01,E682C6A157D0001B0001,Client736,Done,ONTHERUN,1000000,1000000,1000000,1000000,0,...,2024-07-01 11:09:00,2024-07-01 11:09:00,1,0.04,1-5M,5-7 YR,3y-5y,1,0.787991,3.906250e+03
40074,2024-07-01,TRSY_20240701_9910,Client775,Done,"OFFTHERUN,OLDBONDS",3660000,3660000,3660000,3660000,0,...,2024-07-01 15:57:00,2024-07-01 15:57:00,0,0.07,1-5M,15-30 YR,10-30y,-1,3.337784,5.718750e+04
40075,2024-07-01,TRSY_20240701_5800,Client775,Done,ONTHERUN,9630000,9630000,9630000,9630000,0,...,2024-07-01 13:15:00,2024-07-01 13:15:00,0,0.06,5-10M,7-10 YR,7y-10y,-1,0.787774,3.761719e+04
40076,2024-07-01,TRSY_20240701_9909,Client775,Done,ONTHERUN,2300000,2300000,2300000,2300000,0,...,2024-07-01 11:58:00,2024-07-01 11:58:00,0,0.06,1-5M,15-30 YR,10-30y,-1,1.191233,1.347656e+04


## Adjust spread
- define optimal spread
- calculate volume changes
- calculate according pnl

### tier 1, sample

In [63]:
t1 = result1[result1['Tier'] == 'TIER1']

In [65]:
t1.spread.describe()

count    212.000000
mean       1.568069
std        3.092379
min        0.000000
25%        0.391650
50%        0.794520
75%        1.944329
max       42.698548
Name: spread, dtype: float64

In [70]:
# Optimal spread table
optimal_t1_spread = {
    '<18mos': 0.66,
    '18m-2y': 3.04,
    '2y-3y': 3.50,
    '3y-5y': 3.27,
    '5y-7y': 4.24,
    '7y-10y': 5.46,
    '10-30y': 2.00
}

# Map the optimal spread to the dataset
t1['Optimal_Spread'] = t1['MaturityBucket'].map(optimal_t1_spread)

C:\Users\ysnow\AppData\Local\Temp\ipykernel_11452\778451421.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  t1['Optimal_Spread'] = t1['MaturityBucket'].map(optimal_t1_spread)


In [73]:
t1.head(2)

,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,RFQ ClosedTime,AnAutoReplyRFQs,First Response Time (Sec),TradeSizeBucket,Sector,MaturityBucket,Sign,spread,PnL,Optimal_Spread
0,2024-09-30,66FA7D45432800D20002,Client8,Done,OFFTHERUN,100000000,100000000,100000000,100000000,100000000,...,2024-09-30 06:28:00,1,0.03,50+M,5-7 YR,5y-7y,-1,2.354141,1171875.000,4.24
1,2024-09-30,TRSY_20240930_15961,Client9,Done,"OFFTHERUN,DOUBLEOLDS",2100000,2100000,2100000,2100000,2100000,...,2024-09-30 16:00:00,1,0.95,1-5M,7-10 YR,7y-10y,-1,2.303971,24609.375,5.46


In [75]:
test3 = t1[t1['MaturityBucket']=='5y-7y']
test3.head(2)

,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,RFQ ClosedTime,AnAutoReplyRFQs,First Response Time (Sec),TradeSizeBucket,Sector,MaturityBucket,Sign,spread,PnL,Optimal_Spread
0,2024-09-30,66FA7D45432800D20002,Client8,Done,OFFTHERUN,100000000,100000000,100000000,100000000,100000000,...,2024-09-30 06:28:00,1,0.03,50+M,5-7 YR,5y-7y,-1,2.354141,1171875.000,4.24
960,2024-09-27,66F6E619432800000006,Client38,Done,OFFTHERUN,25300000,25300000,25300000,25300000,25300000,...,2024-09-27 13:06:00,1,0.03,25-50M,7-10 YR,5y-7y,-1,0.752276,98828.125,4.24


In [85]:
test3['status'] = test3.apply(lambda row: 'keep' if row['spread']*2 > row['Optimal_Spread'] else 'cancel', axis=1)

C:\Users\ysnow\AppData\Local\Temp\ipykernel_11452\534037092.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test3['status'] = test3.apply(lambda row: 'keep' if row['spread']*2 > row['Optimal_Spread'] else 'cancel', axis=1)


In [86]:
test3.status.value_counts()

status
cancel    24
keep      12
Name: count, dtype: int64

In [87]:
test3[['Client','spread','Optimal_Spread']]

,Client,spread,Optimal_Spread
0,Client8,2.354141,4.24
960,Client38,0.752276,4.24
980,Client603,2.322970,4.24
981,Client603,0.775795,4.24
1588,Client8,3.139594,4.24
1629,Client780,1.477978,4.24
3004,Client603,3.872517,4.24
3663,Client8,0.775705,4.24
3688,Client603,2.918111,4.24
3705,Client780,2.956502,4.24


### all table

In [ ]:
# chao's original off, will be replaced by a table
optimal_spread_data = {
    'MaturityBucket': ['<18mos', '18m-2y', '2y-3y', '3y-5y', '5y-7y', '7y-10y', '10-30y'],
    'TIER1': [0.66, 3.04, 3.50, 3.27, -4.24, 5.46, 2.00],
    'TIER2': [0.74, -3.03, -2.85, -2.36, 3.26, 1.28, 0.75],
    'TIER3': [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    'TIER9': [-0.63, 0.89, 1.71, -0.33, 3.66, 0.98, 3.20],
    'TierGOLD': [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    'TierINTERNAL': [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    'MANUAL': [0.94, -3.12, -2.69, -2.42, 5.71, 5.25, 0.19]
}

optimal_spread_df_off = pd.DataFrame(optimal_spread_data_off)

# Melt the optimal spread table for easier merging
optimal_spread_melted_off = optimal_spread_df.melt_off(
    id_vars=['MaturityBucket'], 
    var_name='Tier', 
    value_name='Optimal_Spread'
)

# Merge the datasets on Tier and Maturity_Bucket
df = result1.merge(optimal_spread_melted_off, on=['Tier', 'MaturityBucket'], how='left')

we accept clients who are willing to pay x times of current spread

In [128]:
df['status'] = df.apply(lambda row: 'keep' if row['spread']*1.5 > row['Optimal_Spread'] else 'cancel', axis=1)

In [129]:
df.status.value_counts()

status
keep      1484
cancel    1367
Name: count, dtype: int64

In [23]:
def pnl_optimal(df):
    # Create a copy of the dataframe
    df_calc = df.copy()
    
    # Initialize optimal_PnL column with zeros
    df_calc['optimal_PnL'] = 0
    
    # Calculate optimal_PnL only for non-cancelled trades
    non_cancelled_mask = df_calc['status'] != 'cancel'
    df_calc.loc[non_cancelled_mask, 'optimal_PnL'] = (
        df_calc['Our Traded Vol (USD)'] *
        df_calc['Sign'] *
        (
            df_calc['Mid Price'] +
            df_calc['Sign'] *
            (
                ((df_calc['Optimal_Spread'] / 2) * df_calc['Deal Value'] / 10000)
            ) -
            df_calc['Mid Price']
        )
    )
    
    # Initialize saving column with zeros
    df_calc['saving'] = 0
    
    # Calculate saving only for non-cancelled trades
    # Note: This uses the existing PnL column without modifying it
    df_calc.loc[non_cancelled_mask, 'saving'] = (
        df_calc.loc[non_cancelled_mask, 'optimal_PnL'] - 
        df_calc.loc[non_cancelled_mask, 'PnL']
    )
    
    return df_calc

In [94]:
def helper(df):
    # Create a copy of the input DataFrame
    df_calc = df.copy()
    
    # Initialize PnL columns
    df_calc['optimal_PnL'] = 0
    
    # Create masks for different spread conditions
    spread_greater = (df_calc['spread'] > df_calc['Optimal_Spread']) & (df_calc['status'] != 'cancel')
    spread_lesser = (df_calc['spread'] <= df_calc['Optimal_Spread']) & (df_calc['status'] != 'cancel')
    
    # Calculate optimal PnL when spread > optimal_spread (use spread)
    df_calc.loc[spread_greater, 'optimal_PnL'] = (
        df_calc.loc[spread_greater, 'Our Traded Vol (USD)'] *
        df_calc.loc[spread_greater, 'Sign'] * (
            df_calc.loc[spread_greater, 'Mid Price'] +
            df_calc.loc[spread_greater, 'Sign'] *
            ((df_calc.loc[spread_greater, 'spread'] / 2) * 
             df_calc.loc[spread_greater, 'Deal Value'] / 10000) -
            df_calc.loc[spread_greater, 'Mid Price']
        )
    )
    
    # Calculate optimal PnL when spread <= optimal_spread (use optimal_spread)
    df_calc.loc[spread_lesser, 'optimal_PnL'] = (
        df_calc.loc[spread_lesser, 'Our Traded Vol (USD)'] *
        df_calc.loc[spread_lesser, 'Sign'] * (
            df_calc.loc[spread_lesser, 'Mid Price'] +
            df_calc.loc[spread_lesser, 'Sign'] *
            ((df_calc.loc[spread_lesser, 'Optimal_Spread'] / 2) * 
             df_calc.loc[spread_lesser, 'Deal Value'] / 10000) -
            df_calc.loc[spread_lesser, 'Mid Price']
        )
    )
    
    # Initialize savings
    df_calc['saving'] = 0
    df_calc.loc[df_calc['status'] != 'cancel', 'saving'] = (
        (df_calc['optimal_PnL'] - df_calc['PnL']).round(2)
    )
    
    return df_calc

In [147]:
df_on = df[df['InstrumentSubGroup']=='ONTHERUN']
df_off = df[df['InstrumentSubGroup']=='OFFTHERUN']

In [148]:
df_on.InstrumentSubGroup.value_counts()

InstrumentSubGroup
ONTHERUN    1995
Name: count, dtype: int64

In [149]:
df_on = helper(df_on)
df_off = helper(df_off)

C:\Users\ysnow\AppData\Local\Temp\ipykernel_58044\1311233633.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 14859.375        7617.22000001  24375.           3544.937
   8007.8125      26656.25        35156.09999997  51328.125
   5844.53125    178962.890625    15625.          24218.75
  76172.20000007  11718.75         5000.         390625.
   5468.75        11640.625       97656.25        78125.
  11443.359375    51679.6875      34453.125        9257.8125
   7968.75         3906.25        26021.37334998 195312.5
   3292.96875     15625.04999998   3906.25        11718.69999999
   5859.375      195312.5        117187.5         18750.
   8046.875       48339.96750003  32031.25       175780.7999999
   3515.64        14648.37499999  39648.4375      25507.8125
  59277.19199997   8554.72400001  39062.5        292970.00000028
  28281.25         4687.5         29296.875       19628.82249998
   1757.835000

In [150]:
df_on

,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,TradeSizeBucket,Sector,MaturityBucket,Sign,spread,PnL,Optimal_Spread,status,optimal_PnL,saving
2,2024-09-30,TRSY_20240930_314,Client9,Done,ONTHERUN,53000000,53000000,53000000,53000000,53000000,...,50+M,7-10 YR,7y-10y,-1,0.783116,207031.2500,5.46,cancel,0.000000,0.00
5,2024-09-30,66FAFF05432800000001,Client164,Done,ONTHERUN,1132000,1132000,1132000,1132000,0,...,1-5M,5-7 YR,5y-7y,1,0.784068,4421.8750,0.80,keep,4511.727500,89.85
6,2024-09-30,66FAEFC145C4000A0001,Client164,Done,ONTHERUN,818000,818000,818000,818000,0,...,500K-1M,7-10 YR,7y-10y,-1,0.783331,3195.3125,0.80,keep,3263.308750,68.00
7,2024-09-30,66FA9F0D45C400400001,Client164,Done,ONTHERUN,500000,500000,500000,500000,0,...,500K-1M,3-5 YR,3y-5y,-1,0.392403,976.5625,0.80,cancel,0.000000,0.00
8,2024-09-30,66FB0D0C45C400000001,Client164,Done,ONTHERUN,500000,500000,500000,500000,500000,...,500K-1M,7-10 YR,7y-10y,1,0.775524,1953.1250,0.80,keep,2014.765626,61.64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2845,2024-07-01,TRSY_20240701_10268,Client690,Done,ONTHERUN,12500000,12500000,12500000,12500000,0,...,10-25M,7-10 YR,7y-10y,-1,0.787154,48828.1250,0.80,keep,49625.000000,796.87
2846,2024-07-01,E682C6A157D0001B0001,Client736,Done,ONTHERUN,1000000,1000000,1000000,1000000,0,...,1-5M,5-7 YR,3y-5y,1,0.787991,3906.2500,2.36,cancel,0.000000,0.00
2848,2024-07-01,TRSY_20240701_5800,Client775,Done,ONTHERUN,9630000,9630000,9630000,9630000,0,...,5-10M,7-10 YR,7y-10y,-1,0.787774,37617.1875,1.28,cancel,0.000000,0.00
2849,2024-07-01,TRSY_20240701_9909,Client775,Done,ONTHERUN,2300000,2300000,2300000,2300000,0,...,1-5M,15-30 YR,10-30y,-1,1.191233,13476.5625,0.75,keep,13476.562500,0.00


In [151]:
df_off

,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,TradeSizeBucket,Sector,MaturityBucket,Sign,spread,PnL,Optimal_Spread,status,optimal_PnL,saving
0,2024-09-30,66FA7D45432800D20002,Client8,Done,OFFTHERUN,100000000,100000000,100000000,100000000,100000000,...,50+M,5-7 YR,5y-7y,-1,2.354141,1.171875e+06,4.24,cancel,0.0000,0.0
3,2024-09-30,66FAEA57432800020001,Client41,Done,OFFTHERUN,500000,500000,500000,500000,500000,...,500K-1M,0-1 YR,<18mos,-1,0.784745,1.953125e+03,0.74,keep,1953.1250,0.0
4,2024-09-30,66FAC35D455C00210006,Client139,Done,OFFTHERUN,1058000,1058000,1058000,1058000,1058000,...,1-5M,5-7 YR,3y-5y,1,1.537811,8.265625e+03,2.36,cancel,0.0000,0.0
14,2024-09-30,TRSY_20240930_4116,Client310,Done,OFFTHERUN,773000,773000,773000,773000,0,...,500K-1M,2-3 YR,<18mos,-1,4.822182,1.811719e+04,0.66,keep,18117.1875,0.0
18,2024-09-30,66FA68EE45C400040002,Client365,Done,OFFTHERUN,5000000,5000000,5000000,5000000,0,...,5-10M,7-10 YR,5y-7y,1,3.539510,7.812500e+04,3.26,keep,78125.0000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2796,2024-07-01,6682E02857D000090001,Client338,Done,OFFTHERUN,30000000,30000000,30000000,30000000,0,...,25-50M,2-3 YR,<18mos,-1,2.375203,3.515625e+05,0.74,keep,351562.5000,0.0
2797,2024-07-01,6682DD8957D000000001,Client362,Done,OFFTHERUN,1000000,1000000,1000000,1000000,0,...,1-5M,2-3 YR,<18mos,-1,2.366584,1.171875e+04,0.74,keep,11718.7500,0.0
2799,2024-07-01,TRSY_20240701_4824,Client406,Done,OFFTHERUN,15000000,15000000,15000000,15000000,0,...,10-25M,0-1 YR,<18mos,-1,2.367237,1.757812e+05,0.74,keep,175781.2500,0.0
2800,2024-07-01,6682EF2145CC00000001,Client426,Done,OFFTHERUN,700000,700000,700000,700000,0,...,500K-1M,7-10 YR,7y-10y,-1,1.637800,5.468750e+03,1.28,keep,5468.7500,0.0


## Condensed Function

In [3]:
def analyze_trades(trade_data, optimal_spread_matrix):
    """
    Analyze trades using optimal spread data to calculate PnL, optimal PnL, and savings.
    
    Parameters:
    trade_data (pd.DataFrame): DataFrame containing trade information with columns:
        - Best Ask Price
        - Best Bid Price
        - Deal Value
        - Our Traded Vol (USD)
        - Sign
        - Mid Price
        - Tier
        - MaturityBucket
        - status
    optimal_spread_matrix (pd.DataFrame): Matrix of optimal spreads with tiers as rows 
        and maturity buckets as columns
    
    Returns:
    pd.DataFrame: Pivot table of savings by tier and maturity bucket
    """
    # Convert optimal spread matrix to long format
    optimal_spread_melted = optimal_spread_matrix.reset_index().melt(
        id_vars=['index'],
        var_name='MaturityBucket',
        value_name='Optimal_Spread'
    ).rename(columns={'index': 'Tier'})
    
    # Calculate initial spread and PnL
    trade_data['spread'] = ((trade_data['Best Ask Price'] - trade_data['Best Bid Price']) / 
                           trade_data['Deal Value']) * 10000
    
    trade_data['PnL'] = (
        trade_data['Our Traded Vol (USD)'] * 
        trade_data['Sign'] * (
            trade_data['Mid Price'] +
            trade_data['Sign'] * ((trade_data['spread'] / 2) * 
                                 trade_data['Deal Value'] / 10000) -
            trade_data['Mid Price']
        )
    )
    
    # Merge with optimal spread data
    df = trade_data.merge(optimal_spread_melted, on=['Tier', 'MaturityBucket'], how='left')
    
    # Calculate optimal PnL based on conditions
    mask_greater = (df['spread'] > df['Optimal_Spread']) & (df['status'] != 'cancel')
    mask_lesser = (df['spread'] <= df['Optimal_Spread']) & (df['status'] != 'cancel')
    
    df['optimal_PnL'] = 0
    
    # For spread > optimal_spread
    df.loc[mask_greater, 'optimal_PnL'] = (
        df.loc[mask_greater, 'Our Traded Vol (USD)'] *
        df.loc[mask_greater, 'Sign'] * (
            df.loc[mask_greater, 'Mid Price'] +
            df.loc[mask_greater, 'Sign'] *
            ((df.loc[mask_greater, 'spread'] / 2) * 
             df.loc[mask_greater, 'Deal Value'] / 10000) -
            df.loc[mask_greater, 'Mid Price']
        )
    )
    
    # For spread <= optimal_spread
    df.loc[mask_lesser, 'optimal_PnL'] = (
        df.loc[mask_lesser, 'Our Traded Vol (USD)'] *
        df.loc[mask_lesser, 'Sign'] * (
            df.loc[mask_lesser, 'Mid Price'] +
            df.loc[mask_lesser, 'Sign'] *
            ((df.loc[mask_lesser, 'Optimal_Spread'] / 2) * 
             df.loc[mask_lesser, 'Deal Value'] / 10000) -
            df.loc[mask_lesser, 'Mid Price']
        )
    )
    
    # Calculate savings
    df['saving'] = 0
    df.loc[df['status'] != 'cancel', 'saving'] = (
        (df['optimal_PnL'] - df['PnL']).round(2)
    )
    
    # Generate summary table
    tiers = ['TIER1', 'TIER2', 'TIER3', 'TIER9', 'TierGOLD', 'TierINTERNAL', 'MANUAL']
    maturity_buckets = ['<18mos', '18m-2y', '2y-3y', '3y-5y', '5y-7y', '7y-10y', '10-30y']
    
    summary = df.pivot_table(
        values='saving',
        index='Tier',
        columns='MaturityBucket',
        aggfunc='sum'
    ).reindex(index=tiers, columns=maturity_buckets)
    
    return summary

# Example usage:
optimal_spread_matrix = pd.DataFrame({
    '<18mos': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    '18m-2y': [3.38, -1.81, -0.07, 0.23, -0.02, -0.52, -2.35],
    '2y-3y': [4.28, -2.31, 0.23, 0.26, 2.00, -0.75, -3.76],
    '3y-5y': [3.24, -1.71, 0.04, -0.43, 0.21, 0.54, -2.23],
    '5y-7y': [-4.65, -3.54, -0.90, 2.59, 0.74, -1.24, 4.99],
    '7y-10y': [4.58, 0.68, 0.57, 1.56, -1.01, 0.20, 2.83],
    '10-30y': [2.00, 0.56, 2.00, 0.52, 1.22, 0.73, 0.32]
}, index=['TIER1', 'TIER2', 'TIER3', 'TIER9', 'TierGOLD', 'TierINTERNAL', 'MANUAL'])

# 2. Use the function
savings_summary = analyze_trades(df, optimal_spread_matrix)

NameError: name 'df' is not defined

In [ ]:
def calculate_saving(df, optimal_spread_df):
    """
    Processes trading data by calculating PnL, optimal PnL, savings, and producing a summary table.

    Args:
    - df: DataFrame containing trading data with necessary columns.
    - optimal_spread_df: DataFrame with optimal spread values by tier and maturity bucket.

    Returns:
    - savings_summary: Pivot table summarizing savings by tier and maturity bucket.
    """
    # Merge the input DataFrame with the optimal spread data
    optimal_spread_melted = optimal_spread_df.melt(
        id_vars=['MaturityBucket'], var_name='Tier', value_name='Optimal_Spread'
    )
    df = df.merge(optimal_spread_melted, on=['Tier', 'MaturityBucket'], how='left')

    # Calculate spread and original PnL
    df['spread'] = ((df['Best Ask Price'] - df['Best Bid Price']) / df['Deal Value']) * 10000
    df['PnL'] = (
        df['Our Traded Vol (USD)'] *
        df['Sign'] * (
            df['Mid Price'] +
            df['Sign'] * ((df['spread'] / 2) * df['Deal Value'] / 10000) -
            df['Mid Price']
        )
    )

    # Initialize optimal PnL and calculate based on spread conditions
    df['optimal_PnL'] = 0
    spread_greater = (df['spread'] > df['Optimal_Spread']) & (df['status'] != 'cancel')
    spread_lesser = (df['spread'] <= df['Optimal_Spread']) & (df['status'] != 'cancel')

    df.loc[spread_greater, 'optimal_PnL'] = (
        df.loc[spread_greater, 'Our Traded Vol (USD)'] *
        df.loc[spread_greater, 'Sign'] * (
            df.loc[spread_greater, 'Mid Price'] +
            df.loc[spread_greater, 'Sign'] *
            ((df.loc[spread_greater, 'spread'] / 2) *
             df.loc[spread_greater, 'Deal Value'] / 10000) -
            df.loc[spread_greater, 'Mid Price']
        )
    )
    df.loc[spread_lesser, 'optimal_PnL'] = (
        df.loc[spread_lesser, 'Our Traded Vol (USD)'] *
        df.loc[spread_lesser, 'Sign'] * (
            df.loc[spread_lesser, 'Mid Price'] +
            df.loc[spread_lesser, 'Sign'] *
            ((df.loc[spread_lesser, 'Optimal_Spread'] / 2) *
             df.loc[spread_lesser, 'Deal Value'] / 10000) -
            df.loc[spread_lesser, 'Mid Price']
        )
    )

    # Calculate savings
    df['saving'] = 0
    df.loc[df['status'] != 'cancel', 'saving'] = (df['optimal_PnL'] - df['PnL']).round(2)

    # Generate the savings summary pivot table
    tiers = ['TIER1', 'TIER2', 'TIER3', 'TIER9', 'TierGOLD', 'TierINTERNAL', 'MANUAL']
    maturity_buckets = ['<18mos', '18m-2y', '2y-3y', '3y-5y', '5y-7y', '7y-10y', '10-30y']
    savings_summary = df.pivot_table(
        values='saving',
        index='Tier',
        columns='MaturityBucket',
        aggfunc='sum'
    ).reindex(index=tiers, columns=maturity_buckets)

    return savings_summary

## Saving

In [ ]:
def saving_summary(df):
    # Define the desired order of tiers and maturity buckets
    tiers = ['TIER1', 'TIER2', 'TIER3', 'TIER9', 'TierGOLD', 'TierINTERNAL', 'MANUAL']
    maturity_buckets = ['<18mos', '18m-2y', '2y-3y', '3y-5y', '5y-7y', '7y-10y', '10-30y']
    
    # Group by tier and maturity_bucket and sum the savings
    pivot_table = df.pivot_table(
        values='saving',  # Column containing the savings values
        index='Tier',      # Your tier column name
        columns='MaturityBucket',  # Your maturity bucket column name
        aggfunc='sum'
    )
    
    # Reorder the rows and columns to match desired order
    pivot_table = pivot_table.reindex(index=tiers, columns=maturity_buckets)
    
    return pivot_table

In [154]:
table = saving_summary(df)
table

MaturityBucket,<18mos,18m-2y,2y-3y,3y-5y,5y-7y,7y-10y,10-30y
Tier,,,,,,,
TIER1,0.00,0.00,136162.11,870937.71,864837.11,107044.74,118146.12
TIER2,0.00,272707.73,557809.07,105673.31,796715.35,116875.89,3124.34
TIER3,0.00,0.00,0.00,71072.51,0.00,81203.13,34409.37
TIER9,0.00,0.00,0.00,0.00,0.00,27283.68,0.00
TierGOLD,0.00,63129.33,5835.94,6410.86,19565.55,91905.31,7381.22
TierINTERNAL,NaN,342.89,591.32,19352.45,3775.28,102761.90,12513.66
MANUAL,66138.39,0.00,0.00,0.00,NaN,0.00,0.00


In [152]:
table_on = saving_summary(df_on)
table_on

MaturityBucket,<18mos,18m-2y,2y-3y,3y-5y,5y-7y,7y-10y,10-30y
Tier,,,,,,,
TIER1,NaN,0.0,0.00,0.00,0.00,0.00,118146.12
TIER2,NaN,0.0,0.00,0.00,0.00,19833.61,3124.34
TIER3,NaN,0.0,0.00,71072.51,NaN,81203.13,34409.37
TIER9,NaN,0.0,NaN,0.00,0.00,27283.68,0.00
TierGOLD,NaN,0.0,5835.94,6410.86,19565.55,91905.31,7381.22
TierINTERNAL,NaN,0.0,0.00,17481.88,3195.22,102761.90,12513.66
MANUAL,NaN,0.0,0.00,0.00,NaN,0.00,0.00


In [153]:
table_off = saving_summary(df_off)
table_off

MaturityBucket,<18mos,18m-2y,2y-3y,3y-5y,5y-7y,7y-10y,10-30y
Tier,,,,,,,
TIER1,0.00,0.00,136162.11,852997.60,864837.11,107044.74,0.0
TIER2,0.00,272707.73,557809.07,30770.55,791497.96,2966.51,0.0
TIER3,0.00,NaN,NaN,NaN,0.00,0.00,NaN
TIER9,0.00,NaN,NaN,0.00,0.00,NaN,0.0
TierGOLD,0.00,NaN,NaN,0.00,NaN,NaN,NaN
TierINTERNAL,NaN,NaN,NaN,0.00,NaN,0.00,NaN
MANUAL,66138.39,NaN,NaN,NaN,NaN,0.00,NaN


In [ ]:
table_on.to_excel('saving_summary.xlsx')

In [137]:
df.to_csv('pnl_table.csv')

## unrealized code

In [ ]:
def analyze_pnl_by_dimension(df):
    analysis = {
        'pnl_by_instrument': df.groupby('InstrumentCode')['PnL'].agg(['sum', 'mean', 'count']),
        'pnl_by_trader': df.groupby('Trader')['PnL'].agg(['sum', 'mean', 'count']),
        'pnl_by_client': df.groupby('Client')['PnL'].agg(['sum', 'mean', 'count']),
        'pnl_by_asset_class': df.groupby('AssetClass')['PnL'].agg(['sum', 'mean', 'count']),
        'daily_pnl': df.groupby(df['EventDate'].dt.date)['PnL'].sum()
    }
    return analysis

In [22]:
df_test = raw.copy()
df_test = pnl(df_test)

In [24]:
result2 = analyze_pnl_by_dimension(df_test)
result2

{'pnl_by_instrument':                      sum          mean  count
 InstrumentCode                               
 912810ES3            0.0      0.000000     10
 912810ET1            0.0      0.000000      9
 912810EV6            0.0      0.000000     13
 912810EW4            0.0      0.000000     10
 912810EX2            0.0      0.000000      5
 ...                  ...           ...    ...
 9999T03Y6            0.0      0.000000     29
 9999T05Y4            0.0      0.000000     25
 9999T07Y2      -244750.0 -10197.916667     24
 9999T10Y7            0.0      0.000000     15
 9999T30Y3            0.0      0.000000      9
 
 [363 rows x 3 columns],
 'pnl_by_trader':                   sum          mean  count
 Trader                                    
 Trader1  9.879443e+06   2342.765770   4217
 Trader2  0.000000e+00      0.000000    154
 Trader3  3.353309e+07  11770.127026   2849
 Trader4  8.665361e+07  12811.000840   6764
 Trader5  2.140044e+07   3627.808354   5899
 Trader6  7.8125

## PnL (inventory control)